# Task 3 - build the DQ reportReads the CSVs you downloaded from Snowsight and writes `03_dq_report.xlsx`.Expected folder layout:```dq_exports/    scorecard.csv    missing_merchant_by_channel.csv    risk_score_by_model.csv    event_lag_by_month.csv    settlement_coverage_by_month.csv    latency_by_provider_month.csv    fanout_proof.csv```

In [ ]:
import pandas as pdfrom pathlib import PathEXPORTS = Path("dq_exports")SHEETS = {    "scorecard":                    "DQ Scorecard",    "missing_merchant_by_channel":  "Missing merchant by channel",    "risk_score_by_model":          "Risk score by model version",    "event_lag_by_month":           "Event lag by month",    "settlement_coverage_by_month": "Settlement coverage by month",    "latency_by_provider_month":    "Latency by provider month",    "fanout_proof":                 "Fan-out proof",}frames = {}for stem, sheet in SHEETS.items():    path = EXPORTS / f"{stem}.csv"    if not path.exists():        print(f"MISSING  {path}")        continue    df = pd.read_csv(path)    df.columns = [c.strip().lower() for c in df.columns]    frames[sheet] = df    print(f"loaded   {path.name:<34} {len(df):>4} rows")

In [ ]:
scorecard = frames["DQ Scorecard"]rank = {"Blocking": 0, "Warning": 1, "Informational": 2}scorecard["_r"] = scorecard["severity"].map(rank).fillna(9)scorecard = (scorecard.sort_values(["_r", "failed_pct"], ascending=[True, False])                      .drop(columns="_r")                      .reset_index(drop=True))frames["DQ Scorecard"] = scorecardsummary = (scorecard.groupby("severity")                    .agg(checks=("check_name", "count"),                         rows_affected=("failed_rows", "sum"))                    .reindex(["Blocking", "Warning", "Informational"])                    .dropna(how="all")                    .reset_index())summary

In [ ]:
from openpyxl import Workbookfrom openpyxl.styles import Font, PatternFill, Alignment, Border, Sidefrom openpyxl.utils.dataframe import dataframe_to_rowsNAVY, ARIAL = "1F3864", "Arial"SEV_FILL = {"Blocking": "F8CBAD", "Warning": "FFE699", "Informational": "E2EFDA"}thin = Side(style="thin", color="BFBFBF")box = Border(left=thin, right=thin, top=thin, bottom=thin)def write_sheet(wb, title, df, widths=None):    ws = wb.create_sheet(title[:31])    for row in dataframe_to_rows(df, index=False, header=True):        ws.append(row)    for c in range(1, df.shape[1] + 1):        cell = ws.cell(row=1, column=c)        cell.font = Font(name=ARIAL, bold=True, size=10, color="FFFFFF")        cell.fill = PatternFill("solid", fgColor=NAVY)        cell.alignment = Alignment(vertical="center", wrap_text=True)        cell.border = box    ws.row_dimensions[1].height = 30    for r in range(2, df.shape[0] + 2):        for c in range(1, df.shape[1] + 1):            cell = ws.cell(row=r, column=c)            cell.font = Font(name=ARIAL, size=9)            cell.alignment = Alignment(vertical="top", wrap_text=True)            cell.border = box    for col, w in (widths or {}).items():        ws.column_dimensions[col].width = w    ws.freeze_panes = "A2"    return wswb = Workbook()wb.remove(wb.active)cover = wb.create_sheet("Cover")cover.sheet_view.showGridLines = Falselines = [    ("AstraPay - Payment Profitability Diagnostic", 18, True),    ("Task 3 - Data Quality Investigation", 13, False),    ("", 10, False),    ("Scope: ASTRAPAY.RAW, 1 January 2025 to 31 December 2025", 10, False),    ("Dimensions: completeness, validity, uniqueness, consistency, timeliness, reconciliation", 10, False),    ("", 10, False),    ("Severity", 11, True),    ("Blocking - corrupts the executive KPI if left untreated.", 10, False),    ("Warning - biases a breakdown but not the headline number.", 10, False),    ("Informational - a valid business state, recorded so it is not mistaken for a defect.", 10, False),    ("", 10, False),    ("Remediation options", 11, True),    ("Exclude, repair, quarantine, or retain with a flag. Each blocking issue carries a decision.", 10, False),]for i, (text, size, bold) in enumerate(lines, start=2):    c = cover.cell(row=i, column=2, value=text)    c.font = Font(name=ARIAL, size=size, bold=bold, color=NAVY if bold else "000000")cover.column_dimensions["A"].width = 3cover.column_dimensions["B"].width = 110ws = write_sheet(wb, "DQ Scorecard", frames["DQ Scorecard"],                 {"A": 15, "B": 46, "C": 20, "D": 12, "E": 12, "F": 11, "G": 14, "H": 58})for r in range(2, frames["DQ Scorecard"].shape[0] + 2):    sev = ws.cell(row=r, column=7).value    if sev in SEV_FILL:        ws.cell(row=r, column=7).fill = PatternFill("solid", fgColor=SEV_FILL[sev])        ws.cell(row=r, column=7).font = Font(name=ARIAL, size=9, bold=True)write_sheet(wb, "Severity Summary", summary, {"A": 18, "B": 12, "C": 16})for sheet, df in frames.items():    if sheet == "DQ Scorecard":        continue    write_sheet(wb, sheet, df, {"A": 28, "B": 18, "C": 18, "D": 18, "E": 18, "F": 18})wb.save("03_dq_report.xlsx")print("wrote 03_dq_report.xlsx with", len(wb.sheetnames), "sheets")

In [ ]:
# quick sanity check against the expected profile of this datasetchecks = {    "transaction.merchant_id is null": (0.5, 1.0),    "fraud_decision.risk_score outside 0 to 1": (30.0, 45.0),    "captured transaction with no settlement row": (3.5, 6.0),}sc = frames["DQ Scorecard"].set_index("check_name")for name, (lo, hi) in checks.items():    if name in sc.index:        v = float(sc.loc[name, "failed_pct"])        flag = "ok" if lo <= v <= hi else "CHECK THE LOAD"        print(f"{name:<48} {v:>7.3f}%   expected {lo}-{hi}%   {flag}")